# PageRank Algorithm

## Learning Objectives

By the end of this notebook you will be able to:

1. **Explain** the random surfer model and why it defines a useful notion of page importance
2. **Define** the transition matrix and why it must be column-stochastic
3. **Identify** spider traps and dead ends and explain why they break the basic algorithm
4. **Derive** the Google Matrix with teleportation and prove it has a unique stationary distribution
5. **Implement** PageRank via power iteration and a memory-efficient sparse variant


## Problem Statement

### The Web as a Directed Graph

The World Wide Web can be modelled as a directed graph:

- **Nodes** = web pages
- **Edges** = hyperlinks (A → B means page A links to page B)

**Question:** Given billions of pages, which ones are most *important* or *authoritative*?

### Naïve idea: in-degree

Count the number of pages linking to a page. Problem: easy to manipulate — spammers create thousands of fake pages pointing to their target.

### PageRank Insight

A link from an important page should count more than a link from an obscure page. Importance must be defined *recursively*:

$$\text{PR}(v) \propto \sum_{u \to v} \frac{\text{PR}(u)}{\text{out-degree}(u)}$$

This is equivalent to the *stationary distribution* of a random walk on the graph.

### The Random Surfer Model

Imagine a web surfer who:
- With probability $\beta$ (e.g. 0.85): follows a uniformly random outgoing link from the current page
- With probability $1 - \beta$: teleports to a uniformly random page in the entire web

The **PageRank** of a page $v$ is the long-run fraction of time the random surfer spends at $v$.


In [ ]:
from IPython.display import SVG, display

svg = '''
<svg xmlns="http://www.w3.org/2000/svg" width="820" height="400" font-family="monospace" font-size="12">

  <!-- background -->
  <rect width="820" height="400" fill="#fafafa" rx="8"/>

  <defs>
    <marker id="arr" markerWidth="9" markerHeight="7" refX="8" refY="3.5" orient="auto">
      <polygon points="0 0, 9 3.5, 0 7" fill="#666"/>
    </marker>
    <marker id="arrteal" markerWidth="9" markerHeight="7" refX="8" refY="3.5" orient="auto">
      <polygon points="0 0, 9 3.5, 0 7" fill="#76b7b2"/>
    </marker>
  </defs>

  <!-- ═══ LEFT: Web Graph ═══ -->
  <text x="185" y="24" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">Web Graph (4 pages)</text>

  <!-- Nodes -->
  <!-- A at (100, 110) -->
  <circle cx="100" cy="110" r="28" fill="#4e79a7" stroke="#fff" stroke-width="2"/>
  <text x="100" y="106" text-anchor="middle" fill="white" font-size="13" font-weight="bold">A</text>
  <text x="100" y="122" text-anchor="middle" fill="white" font-size="10">PR=0.37</text>

  <!-- B at (280, 60) -->
  <circle cx="280" cy="60" r="22" fill="#f28e2b" stroke="#fff" stroke-width="2"/>
  <text x="280" y="56" text-anchor="middle" fill="white" font-size="13" font-weight="bold">B</text>
  <text x="280" y="72" text-anchor="middle" fill="white" font-size="10">PR=0.19</text>

  <!-- C at (280, 175) -->
  <circle cx="280" cy="175" r="22" fill="#59a14f" stroke="#fff" stroke-width="2"/>
  <text x="280" y="171" text-anchor="middle" fill="white" font-size="13" font-weight="bold">C</text>
  <text x="280" y="187" text-anchor="middle" fill="white" font-size="10">PR=0.29</text>

  <!-- D at (140, 250) -->
  <circle cx="140" cy="250" r="18" fill="#e15759" stroke="#fff" stroke-width="2"/>
  <text x="140" y="246" text-anchor="middle" fill="white" font-size="13" font-weight="bold">D</text>
  <text x="140" y="262" text-anchor="middle" fill="white" font-size="10">PR=0.15</text>

  <!-- Edges (directed) -->
  <!-- A -> B -->
  <line x1="126" y1="95" x2="258" y2="67" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- A -> C -->
  <line x1="126" y1="123" x2="258" y2="168" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- B -> C -->
  <line x1="280" y1="82" x2="280" y2="153" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- C -> A -->
  <line x1="258" y1="162" x2="125" y2="128" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- D -> A -->
  <line x1="122" y1="235" x2="107" y2="138" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- D -> B -->
  <line x1="157" y1="238" x2="262" y2="73" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- A -> D -->
  <line x1="80" y1="130" x2="125" y2="238" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>

  <!-- Edge labels -->
  <text x="180" y="70"  fill="#555" font-size="10">1/3</text>
  <text x="165" y="158" fill="#555" font-size="10">1/3</text>
  <text x="286" y="120" fill="#555" font-size="10">1</text>
  <text x="178" y="148" fill="#555" font-size="10">1</text>
  <text x="87"  y="192" fill="#555" font-size="10">1/3</text>
  <text x="74"  y="182" fill="#555" font-size="10">A→D</text>

  <!-- ═══ MIDDLE: Transition Matrix ═══ -->
  <text x="460" y="24" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">Column-stochastic Matrix M</text>

  <!-- matrix border -->
  <rect x="380" y="35" width="165" height="130" rx="4" fill="none" stroke="#ccc" stroke-width="1"/>

  <!-- header row -->
  <text x="420" y="52" text-anchor="middle" fill="#555" font-size="11">from→</text>
  <text x="460" y="52" text-anchor="middle" fill="#4e79a7" font-size="11" font-weight="bold">A</text>
  <text x="500" y="52" text-anchor="middle" fill="#f28e2b" font-size="11" font-weight="bold">B</text>
  <text x="535" y="52" text-anchor="middle" fill="#59a14f" font-size="11" font-weight="bold">C</text>

  <!-- row A -->
  <text x="395" y="72" fill="#4e79a7" font-size="11" font-weight="bold">A</text>
  <text x="460" y="72" text-anchor="middle" fill="#333" font-size="11">0</text>
  <text x="500" y="72" text-anchor="middle" fill="#333" font-size="11">0</text>
  <text x="535" y="72" text-anchor="middle" fill="#333" font-size="11">1</text>

  <!-- row B -->
  <text x="395" y="96" fill="#f28e2b" font-size="11" font-weight="bold">B</text>
  <text x="460" y="96" text-anchor="middle" fill="#333" font-size="11">1/3</text>
  <text x="500" y="96" text-anchor="middle" fill="#333" font-size="11">0</text>
  <text x="535" y="96" text-anchor="middle" fill="#333" font-size="11">0</text>

  <!-- row C -->
  <text x="395" y="120" fill="#59a14f" font-size="11" font-weight="bold">C</text>
  <text x="460" y="120" text-anchor="middle" fill="#333" font-size="11">1/3</text>
  <text x="500" y="120" text-anchor="middle" fill="#333" font-size="11">1</text>
  <text x="535" y="120" text-anchor="middle" fill="#333" font-size="11">0</text>

  <!-- row D -->
  <text x="395" y="144" fill="#e15759" font-size="11" font-weight="bold">D</text>
  <text x="460" y="144" text-anchor="middle" fill="#333" font-size="11">1/3</text>
  <text x="500" y="144" text-anchor="middle" fill="#333" font-size="11">0</text>
  <text x="535" y="144" text-anchor="middle" fill="#333" font-size="11">0</text>

  <text x="460" y="180" text-anchor="middle" fill="#555" font-size="10">columns sum to 1 (stochastic)</text>

  <!-- ═══ MIDDLE: Google Matrix ═══ -->
  <text x="460" y="210" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Google Matrix G = βM + (1−β)ee^T/n</text>
  <text x="460" y="228" text-anchor="middle" fill="#555" font-size="11">β = 0.85  (damping factor)</text>
  <text x="460" y="246" text-anchor="middle" fill="#555" font-size="11">Teleportation probability = 0.15/4 = 0.0375</text>

  <!-- ═══ RIGHT: Power Iteration convergence ═══ -->
  <text x="680" y="24" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">Power Iteration Convergence</text>

  <!-- axes -->
  <line x1="580" y1="290" x2="790" y2="290" stroke="#333" stroke-width="1.5"/>
  <line x1="580" y1="290" x2="580" y2="60"  stroke="#333" stroke-width="1.5"/>
  <polygon points="790,286 790,294 800,290" fill="#333"/>
  <polygon points="576,60 584,60 580,50"   fill="#333"/>
  <text x="800" y="294" fill="#555" font-size="10">iter</text>
  <text x="568" y="50"  fill="#555" font-size="10">PR</text>

  <!-- tick labels x: 0,5,10,15,20 -->
  <line x1="580" y1="290" x2="580" y2="297" stroke="#888" stroke-width="1"/>
  <line x1="632" y1="290" x2="632" y2="297" stroke="#888" stroke-width="1"/>
  <line x1="685" y1="290" x2="685" y2="297" stroke="#888" stroke-width="1"/>
  <line x1="737" y1="290" x2="737" y2="297" stroke="#888" stroke-width="1"/>
  <line x1="790" y1="290" x2="790" y2="297" stroke="#888" stroke-width="1"/>
  <text x="580" y="308" text-anchor="middle" fill="#555" font-size="10">0</text>
  <text x="632" y="308" text-anchor="middle" fill="#555" font-size="10">5</text>
  <text x="685" y="308" text-anchor="middle" fill="#555" font-size="10">10</text>
  <text x="737" y="308" text-anchor="middle" fill="#555" font-size="10">15</text>
  <text x="790" y="308" text-anchor="middle" fill="#555" font-size="10">20</text>

  <!-- y ticks: 0, 0.1, 0.2, 0.3, 0.4 -->
  <text x="576" y="290" text-anchor="end" fill="#555" font-size="10">0</text>
  <text x="576" y="233" text-anchor="end" fill="#555" font-size="10">.1</text>
  <text x="576" y="176" text-anchor="end" fill="#555" font-size="10">.2</text>
  <text x="576" y="119" text-anchor="end" fill="#555" font-size="10">.3</text>
  <text x="576" y="62"  text-anchor="end" fill="#555" font-size="10">.4</text>

  <!-- PR(A) converges to ~0.37 — polyline, x step=10.5/iter -->
  <!-- Start at 0.25 each, converge to A=0.37, B=0.19, C=0.29, D=0.15 -->
  <!-- A: 0.25 -> 0.30 -> 0.34 -> 0.36 -> 0.37 -> 0.37 -->
  <polyline points="580,148 601,125 632,101 685,91 737,87 790,86" fill="none" stroke="#4e79a7" stroke-width="2.5"/>
  <!-- B: 0.25 -> 0.22 -> 0.20 -> 0.19 -> 0.19 -->
  <polyline points="580,148 601,159 632,167 685,172 737,173 790,174" fill="none" stroke="#f28e2b" stroke-width="2.5"/>
  <!-- C: 0.25 -> 0.27 -> 0.28 -> 0.29 -> 0.29 -->
  <polyline points="580,148 601,138 632,127 685,119 737,118 790,118" fill="none" stroke="#59a14f" stroke-width="2.5"/>
  <!-- D: 0.25 -> 0.21 -> 0.17 -> 0.16 -> 0.15 -->
  <polyline points="580,148 601,162 632,179 685,190 737,192 790,194" fill="none" stroke="#e15759" stroke-width="2.5"/>

  <!-- legend -->
  <line x1="600" y1="330" x2="620" y2="330" stroke="#4e79a7" stroke-width="2.5"/>
  <text x="625" y="334" fill="#555" font-size="10">A (0.37)</text>
  <line x1="680" y1="330" x2="700" y2="330" stroke="#f28e2b" stroke-width="2.5"/>
  <text x="705" y="334" fill="#555" font-size="10">B (0.19)</text>
  <line x1="600" y1="348" x2="620" y2="348" stroke="#59a14f" stroke-width="2.5"/>
  <text x="625" y="352" fill="#555" font-size="10">C (0.29)</text>
  <line x1="680" y1="348" x2="700" y2="348" stroke="#e15759" stroke-width="2.5"/>
  <text x="705" y="352" fill="#555" font-size="10">D (0.15)</text>

</svg>
'''

display(SVG(svg))


## The Transition Matrix

### Column-Stochastic Form

Let $n$ be the number of pages. Define the **transition matrix** $M \in \mathbb{R}^{n \times n}$:

$$M_{ij} = \begin{cases} \frac{1}{d_j} & \text{if there is a link } j \to i \\ 0 & \text{otherwise} \end{cases}$$

where $d_j$ is the out-degree of node $j$.

**Key property:** each column of $M$ sums to 1 (column-stochastic). The $j$-th column distributes page $j$'s "importance vote" equally among its out-links.

### Power Iteration (Basic)

Start with any distribution $r^{(0)} = \mathbf{1}/n$. Iterate:

$$r^{(t+1)} = M\, r^{(t)}$$

If $M$ is column-stochastic and the graph is strongly connected and aperiodic, $r^{(t)} \to r^*$ (the stationary distribution) as $t \to \infty$.

### Problems

| Problem | Cause | Symptom |
|---------|-------|---------|
| **Spider trap** | Group of nodes with no outgoing edges to the rest | Absorbs all PageRank mass |
| **Dead end** | Node with no out-links | Leaks mass (column sums < 1) |

Both violate the ergodicity conditions needed for a unique stationary distribution.


## The Google Matrix

### Fix for Dead Ends

For a dead-end node $j$ (no out-links), distribute its entire mass uniformly to all pages:

$$M_{ij}^{\text{fixed}} = \frac{1}{n} \quad \text{for all } i, \text{ if } d_j = 0$$

This makes $M^{\text{fixed}}$ column-stochastic.

### Fix for Spider Traps — Teleportation

Introduce a **damping factor** $\beta \in (0, 1)$:

$$G = \beta M^{\text{fixed}} + (1-\beta)\frac{1}{n}\mathbf{1}\mathbf{1}^\top$$

where $\mathbf{1}\mathbf{1}^\top/n$ is the matrix where every entry equals $1/n$.

**Interpretation:** with probability $\beta$ follow an out-link; with probability $1-\beta$ teleport to a random page.

### Perron-Frobenius Theorem

Because $G$ is:
- **Stochastic** (columns sum to 1)
- **Primitive** (all entries $> 0$ due to teleportation — irreducible + aperiodic)

The Perron-Frobenius theorem guarantees $G$ has a **unique** stationary distribution $r^*$ with $G r^* = r^*$.

### Power Iteration on $G$

$$r^{(t+1)} = G\, r^{(t)} = \beta M^{\text{fixed}} r^{(t)} + \frac{1-\beta}{n}\mathbf{1}$$

We never store $G$ explicitly (too large). We apply it as:

$$r^{(t+1)} = \beta \bigl(M^{\text{fixed}} r^{(t)}\bigr) + \frac{1-\beta}{n}$$

**Convergence:** the second-largest eigenvalue of $G$ is at most $\beta$, so:

$$\| r^{(t)} - r^* \|_1 \leq \beta^t \| r^{(0)} - r^* \|_1$$

For $\beta = 0.85$, convergence within 1% error requires $\approx 57$ iterations.


## Algorithm Steps

### Inputs

- Directed graph $G$ with $n$ nodes, edges as adjacency list
- Damping factor $\beta$ (typically 0.85)
- Convergence tolerance $\epsilon$ (typically $10^{-8}$)

### Steps

1. **Build** column-stochastic matrix $M$ from adjacency list; handle dead ends uniformly

2. **Initialise** $r = \mathbf{1}/n$ (uniform distribution)

3. **Iterate** until $\|r^{(t+1)} - r^{(t)}\|_1 < \epsilon$:

   $$r \leftarrow \beta (M r) + \frac{1-\beta}{n}\mathbf{1}$$

4. **Output** $r^*$ — the PageRank vector

### Complexity per Iteration

- Dense matrix: $O(n^2)$ — infeasible for the web
- Sparse matrix: $O(|E|)$ where $|E|$ = number of edges — feasible
- Memory: $O(n + |E|)$ — store out-link lists plus two rank vectors

### Number of Iterations

Power iteration converges geometrically with ratio $\beta$. In practice 50–100 iterations suffice for $\beta = 0.85$.


In [ ]:
import numpy as np


def build_transition_matrix(adj):
    """
    Build the column-stochastic transition matrix M from an adjacency list.

    Inputs
    ------
    adj : dict {node: list_of_neighbours} — directed edges out of each node

    Output
    ------
    M    : np.ndarray shape (n, n) — column-stochastic; M[i,j] = prob of moving to i from j
    nodes: list — node ordering (index -> node)
    """
    nodes = sorted(adj.keys())
    n = len(nodes)
    idx = {node: i for i, node in enumerate(nodes)}

    M = np.zeros((n, n))
    for src, neighbours in adj.items():
        if neighbours:
            # Each out-link carries equal probability
            prob = 1.0 / len(neighbours)
            for dst in neighbours:
                M[idx[dst], idx[src]] = prob
        else:
            # Dead end: teleport uniformly (dangling node fix)
            M[:, idx[src]] = 1.0 / n

    return M, nodes


def pagerank(adj, beta=0.85, max_iter=100, tol=1e-8):
    """
    Compute PageRank via power iteration on the Google Matrix.

    Inputs
    ------
    adj     : dict {node: list_of_neighbours}
    beta    : float — damping factor (0.85 is standard)
    max_iter: int — maximum power iterations
    tol     : float — convergence threshold (L1 norm of change)

    Output
    ------
    pr      : dict {node: pagerank_score} — scores sum to 1
    n_iters : int — iterations until convergence
    """
    M, nodes = build_transition_matrix(adj)
    n = len(nodes)

    # Google matrix: G = beta * M + (1-beta) * (1/n) * ones
    # We never form G explicitly — cheaper to apply it as: G@r = beta*M@r + (1-beta)/n
    teleport = (1.0 - beta) / n

    # Initialise to uniform distribution
    r = np.ones(n) / n

    for i in range(max_iter):
        r_new = beta * (M @ r) + teleport
        # r_new already sums to 1 if M is column-stochastic and r sums to 1

        if np.sum(np.abs(r_new - r)) < tol:
            r = r_new
            return {nodes[j]: float(r[j]) for j in range(n)}, i + 1
        r = r_new

    return {nodes[j]: float(r[j]) for j in range(n)}, max_iter


def pagerank_sparse(edges, n_nodes, beta=0.85, max_iter=100, tol=1e-8):
    """
    Memory-efficient PageRank for large sparse graphs.
    Uses out-degree list instead of full matrix.

    Inputs
    ------
    edges   : list of (src, dst) integer tuples
    n_nodes : int — total number of nodes (0..n_nodes-1)
    """
    out_links  = [[] for _ in range(n_nodes)]
    out_degree = np.zeros(n_nodes, dtype=int)
    for src, dst in edges:
        out_links[src].append(dst)
        out_degree[src] += 1

    teleport = (1.0 - beta) / n_nodes
    r = np.ones(n_nodes) / n_nodes

    for iteration in range(max_iter):
        r_new = np.zeros(n_nodes)
        dangling_sum = 0.0

        for src in range(n_nodes):
            if out_degree[src] == 0:
                # Dangling node: distribute its mass uniformly
                dangling_sum += r[src]
            else:
                contribution = beta * r[src] / out_degree[src]
                for dst in out_links[src]:
                    r_new[dst] += contribution

        # Add dangling mass and teleportation uniformly
        r_new += (beta * dangling_sum / n_nodes) + teleport

        if np.sum(np.abs(r_new - r)) < tol:
            return r_new, iteration + 1
        r = r_new

    return r, max_iter


# ── Demo ──────────────────────────────────────────────────────────────────────
adj = {
    "A": ["B", "C", "D"],
    "B": ["C"],
    "C": ["A"],
    "D": ["A", "B"],
}

pr, n_iters = pagerank(adj, beta=0.85)
print(f"Converged in {n_iters} iterations")
print(f"\nPageRank scores (sum = {sum(pr.values()):.6f}):")
for node, score in sorted(pr.items(), key=lambda x: -x[1]):
    bar = '█' * int(score * 60)
    print(f"  {node}: {score:.4f}  {bar}")


In [ ]:
import numpy as np


def demo_spider_trap():
    """Show how a spider trap absorbs all PageRank without teleportation."""
    # Graph: A->B, B->A (spider trap), C->A, D->C
    # Without beta, A and B together absorb all mass.
    adj_trap = {
        "A": ["B"],
        "B": ["A"],
        "C": ["A"],
        "D": ["C"],
    }

    # Without teleportation (beta=1.0)
    from collections import defaultdict
    nodes = sorted(adj_trap.keys())
    n = len(nodes)
    idx = {node: i for i, node in enumerate(nodes)}
    M = np.zeros((n, n))
    for src, nbrs in adj_trap.items():
        prob = 1.0 / len(nbrs)
        for dst in nbrs:
            M[idx[dst], idx[src]] = prob

    r = np.ones(n) / n
    for _ in range(200):
        r = M @ r

    print("Without teleportation (beta=1) — spider trap absorbs everything:")
    for i, node in enumerate(nodes):
        print(f"  {node}: {r[i]:.4f}")

    # With teleportation (beta=0.85)
    pr, iters = pagerank(adj_trap, beta=0.85)
    print(f"\nWith teleportation (beta=0.85) — converged in {iters} iters:")
    for node, score in sorted(pr.items(), key=lambda x: -x[1]):
        print(f"  {node}: {score:.4f}")


demo_spider_trap()
